# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Fediv Andrii'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-34'
НОМЕР        = 13       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 13   ·   Fediv Andrii   ·   ШІ-34

Ваші вісім стовпців:
   1. номер і/х
   2. Діагноз
   3. ЛН
   4. Дата розтину
   5. КТ
   6. Прокальцитонін
   7. БРзаг2
   8. МНВ1

Ваша пара для аналізу зв’язку: Перебіг та ступінь КТ

Ваше додаткове питання:
   Стовпець «помер/жив» містить числа 1 і 2. Які три різні статистики pandas порахує для нього без жодної скарги — і чому кожна з них безглузда?


In [3]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,номер і/х,Діагноз,ЛН,Дата розтину,КТ,Прокальцитонін,БРзаг2,МНВ1
0,2377/239,Негос дв полісегм пнев 4 кл гр,ІІІ-IV,2022-03-11,20-30%,0.126,17.0,3.10
1,1121/19,Негосп полісег пневм 4 кл гр,ІІ-ІІІ,NaT,30-35%,NaN,13.6,1.22
2,283/674,Негос полісегм пнев 4 кл гр,ІІ,NaT,70-80%,NaN,18.7,1.03
3,2372/513,Негос дв полісегм пнев 4 кл гр,ІІІ-IV,NaT,0.8,NaN,13.6,1.69
4,1441/671,Негос дв полісегм пнев 4 кл гр,ІІ-ІІІ,NaT,NaN,NaN,15.3,1.15
5,2573/354,Негос дв полісегм пнев 4 кл гр,ІІ-ІІІ,2021-12-26,30-40%,NaN,15.3,0.90
6,1762/555,Негос дв полісегм пнев 4 кл гр,ІІ-ІІІ,NaT,60-65%,NaN,15.3,1.09
7,2664/348,Негос дв полісегм пнев 3 кл гр,І,NaT,до 17%,NaN,15.3,0.94


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [4]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   номер і/х       121 non-null    object        
 1   Діагноз         121 non-null    object        
 2   ЛН              121 non-null    object        
 3   Дата розтину    61 non-null     datetime64[ns]
 4   КТ              44 non-null     object        
 5   Прокальцитонін  46 non-null     float64       
 6   БРзаг2          101 non-null    float64       
 7   МНВ1            90 non-null     float64       
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 7.7+ KB


In [5]:
# Далі — що є всередині насправді
for c in МОЇ:
    nna = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in nna.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(nna)} з {len(df)} | унікальних {nna.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in nna.unique()[:10]]}')
    print()

--- номер і/х
    непорожніх 121 з 121 | унікальних 119 | типи: int×65, str×56
    приклади: ["'2377/239'", "'1121/19'", "'283/674'", "'2372/513'", "'1441/671'", "'2573/354'", "'1762/555'", "'2664/348'", "'1806/185'", "'1717/25'"]

--- Діагноз
    непорожніх 121 з 121 | унікальних 15 | типи: str×121
    приклади: ["'Негос дв полісегм пнев 4 кл гр '", "'Негосп полісег пневм 4 кл гр'", "'Негос полісегм пнев 4 кл гр'", "'Негос дв полісегм пнев 3 кл гр '", "'Негосп полісег пневм 3 кл гр'", "'Негосп полісег пнев 4 кл гр'", "'Негосп полісегм пнев 3 кл гр'", "'Негосп дв полісегм пнев 3 кл гр'", "'Негосп дв полісегм пнев 4 кл гр'", "'Негосп дв полісегм пнев 4 кл група'"]

--- ЛН
    непорожніх 121 з 121 | унікальних 6 | типи: str×121
    приклади: ["'ІІІ-IV'", "'ІІ-ІІІ'", "'ІІ'", "'І'", "'ІІІ'", "'І-ІІ'"]

--- Дата розтину
    непорожніх 61 з 121 | унікальних 50 | типи: Timestamp×61
    приклади: ["Timestamp('2022-03-11 00:00:00')", "Timestamp('2021-12-26 00:00:00')", "Timestamp('2021-11-19 00

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [6]:
ПАСПОРТ = {
    "номер і/х": ("ідентифікатор", "Номер історії хвороби лише позначає пацієнта. також є дублікати"),
    "Діагноз": (
        "номінальна",
        "Текстова назва діагнозу; категорії можна лише порівнювати на рівність. 15 унікальних значень це різні написання 2–4 діагнозів.",
    ),
    "ЛН": (
        "порядкова",
        "Ступінь легеневої недостатності І < ІІ має природний порядок, але відстань між сусідніми ступенями невідома",
    ),
    "Дата розтину": (
        "інтервальна",
        "Дата — точка на шкалі часу з нулем у початку епохи, різниці дат мають зміст, але відношення дат немає сенсу",
    ),
    "КТ": (
        "відношень",
        "Компютерна томографія легень??? ураження? має нуль, відноення мають значення; є проблема у змішаному записі (0.3, 25-30%, до 17%).",
    ),
    "Прокальцитонін": (
        "відношень",
        "має фізичний нуль — відсутність речовини, відношення значень теж мають сенс",
    ),
    "БРзаг2": (
        "відношень",
        "Загальний білірубін (мкмоль/л)и тн57го — концентрація з природним нулем, допустимі і різниці, і відношення, а одиниця вимірювання умовна ",
    ),
    "МНВ1": (
        "відношень",
        "Міжнародне нормалізоване відношення нуль природний, те що відношення ок, каже сама назва.",
    ),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ("", ""))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ], columns=["Стовпець", "Шкала", "Обґрунтування"]
)
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,номер і/х,ідентифікатор,Номер історії хвороби лише позначає пацієнта. також є ду...
1,Діагноз,номінальна,Текстова назва діагнозу; категорії можна лише порівнюват...
2,ЛН,порядкова,Ступінь легеневої недостатності І < ІІ має природний пор...
3,Дата розтину,інтервальна,"Дата — точка на шкалі часу з нулем у початку епохи, різн..."
4,КТ,відношень,"Компютерна томографія легень??? ураження? має нуль, відн..."
5,Прокальцитонін,відношень,"має фізичний нуль — відсутність речовини, відношення зна..."
6,БРзаг2,відношень,Загальний білірубін (мкмоль/л)и тн57го — концентрація з ...
7,МНВ1,відношень,"Міжнародне нормалізоване відношення нуль природний, те щ..."


In [7]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    shcala, об = ПАСПОРТ[c]
    assert shcala.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{shcala}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [8]:
from pandas import isna


dataset = df[МОЇ]


def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    nna = dataset[стовпець].dropna()
    p = {"непорожніх": len(nna), "унікальних": nna.nunique()}
    shcala = шкала.strip().lower()

    p["мода"] = nna.mode()
    p["частоти"] = nna.value_counts().head(20)

    if shcala in ("ідентифікатор", "номінальна"):
        return p

    p["медіана"] = nna.median()
    p["квантилі"] = nna.quantile()

    if shcala in ("порядкова"):
        return p

    p["середнє"] = nna.mean()
    p["стандартне відхилення"] = nna.std()

    if shcala in ("інтервальна"):
        return p

    p["коефіцієнт варіації"] = p["стандартне відхилення"] / p["середнє"]

    # ваш код: додайте моду й частоти для всіх шкал
    # для порядкової — медіану та квантилі
    # для інтервальної та вище — середнє й стандартне відхилення
    # для відношень і абсолютної — коефіцієнт варіації
    return p


pd.set_option("display.max_rows", None)


def normalize_percent(value):
    x = str(value).strip().replace("%", "")

    if not x or x == "nan":
        return np.nan

    if "-" in x:
        a, b = map(float, x.split("-"))
        return (a + b) / 200

    if "до " in x:
        return float(x.replace("до ", "")) / 100

    return float(x) if float(x) <= 1 else float(x) / 100


dataset["ЛН"] = dataset["ЛН"].map({"ІІІ-IV": 3.5, "ІІ-ІІІ": 2.5, "ІІ": 2, "І": 1, "ІІІ": 3, "І-ІІ": 1.5})
dataset["КТ"] = dataset["КТ"].transform(normalize_percent)
for c in МОЇ:
    print(f"--- {c}  ({ПАСПОРТ[c][0]})")
    print("   ", допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- номер і/х  (ідентифікатор)
    {'непорожніх': 121, 'унікальних': 119, 'мода': 0    1837
1    2526
Name: номер і/х, dtype: object, 'частоти': номер і/х
2526        2
1837        2
2377/239    1
2372/513    1
1441/671    1
1121/19     1
283/674     1
2664/348    1
1806/185    1
1717/25     1
3213/476    1
2831/393    1
2998/135    1
2673/681    1
1731/280    1
2612/542    1
3072/418    1
2592/430    1
1276/596    1
2501/673    1
Name: count, dtype: int64}

--- Діагноз  (номінальна)
    {'непорожніх': 121, 'унікальних': 15, 'мода': 0    Негосп дв полісегм пнев 4 кл група
Name: Діагноз, dtype: object, 'частоти': Діагноз
Негосп дв полісегм пнев 4 кл група         43
НГ дв. Полісег пневмонія 4кл гр.           19
Негосп дв полісегм пнев 3 кл гр            14
НГ дв. Полісег пневмонія 3 кл.гр            9
Негос дв полісегм пнев 4 кл гр              8
Негосп дв полісегм пневмонія 4 кл група     7
Негосп дв полісегм пнев 4 кл гр             6
Негос дв полісегм пнев 3 кл гр              3
Него

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [9]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
номер і/х,121.0,119.0,2526.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Діагноз,121,15,Негосп дв полісегм пнев 4 кл група,43,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ЛН,121,6,ІІ-ІІІ,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Дата розтину,61,NaN,NaN,NaN,2022-04-06 13:22:37.377049088,2021-01-06 00:00:00,2022-03-29 00:00:00,2022-04-23 00:00:00,2022-05-07 00:00:00,2022-06-22 00:00:00,NaN
КТ,44.0,26.0,0.3,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Прокальцитонін,46.0,NaN,NaN,NaN,0.752957,0.043,0.1,0.1,0.1205,18.22,3.010375
БРзаг2,101.0,NaN,NaN,NaN,16.216832,7.4,13.6,15.3,18.7,60.5,5.537094
МНВ1,90.0,NaN,NaN,NaN,1.333778,0.8,1.0525,1.19,1.34,10.6,1.032131


In [10]:
БЕЗГЛУЗДІ = [
    ('ЛН', 'середнє', dataset['ЛН'].dropna().mean(), 'середнє по ступені легеневої недостатності немає сенсу через те, і є неефективним, краще мода або хоча б медіана'), 
    ('Дата розтину', 'середнє', dataset['Дата розтину'].dropna().mean(), 'немає сенсу обрахунку середньої дати розтину, це не дасть нам розуміння коли всередньому відбувався розтин'), 
    ('Діагноз', 'freq', dataset['Діагноз'].value_counts()[0], 'зважаючи на однотипність діагнозів сенсу із того що нам 43 рази попажався Негосп дв полісегм пнев 4 кл група практично 0'), 
    # ('стовпець', 'яка статистика', значення, 'чому це число нічого не означає'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,ЛН,середнє,2.285124,середнє по ступені легеневої недостатності немає сенсу ч...
1,Дата розтину,середнє,2022-04-06 13:22:37.377049088,"немає сенсу обрахунку середньої дати розтину, це не даст..."
2,Діагноз,freq,43,зважаючи на однотипність діагнозів сенсу із того що нам ...


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [11]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(10).to_string())

Ваша пара: Перебіг × ступінь КТ

Перебіг: шкала — не у вашому списку
ступінь КТ: шкала — не у вашому списку

     Перебіг ступінь КТ
0     важкий    сердній
1     важкий   середній
2     важкий  критичний
3     важкий  критичний
4     важкий        NaN
5     важкий   середній
6     важкий     важкий
7  серед важ     легкий
8     важкий        NaN
9     важкий  критичний


In [12]:
# Перебіг - порядкова
# Ступінь КТ - порядкова

print(df["Перебіг"].unique(), df["ступінь КТ"].unique())

ОБРАНА_МІРА = "рангова кореляція Спірмена"  # напишіть назву міри
ЧОМУ_САМЕ_ВОНА = "через те що 1. так написано в табличці 2. Поточний метод вразовує саме порядковість шкал не враховуюючи відстані ну і 3. якби Обрати Спірмана в поточній ситуації, то через нерівність кількості категорій у двох колонках він би дав безглузді значення"  # одне-два речення

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, "Заповніть обидва поля"

dataset["Перебіг"] = df["Перебіг"].map({"серед важ": 1, "важкий": 2, "вкрай важкий ": 3})
dataset["ступінь КТ"] = df["ступінь КТ"].map(
    {
        "сердній": 2,
        "середній": 2,
        "критичний": 4,
        "важкий": 3,
        "легкий": 1,
        "серед ступ": 2,
        "легка": 1,
        "вкрай важкий ": 5,
        "легкий ": 1,
        "вкрай важкий": 5,
        "важкий ": 3,
        "серед ступеня": 2,
    }
)

dataSpirman = dataset[["Перебіг", "ступінь КТ"]].dropna()
n = len(dataSpirman)
# ранк ставить у порядку зростання ранги а однаковим елементам їх середній ранг м
spirmam = 1 - 6 * sum((dataSpirman["Перебіг"].rank() - dataSpirman["ступінь КТ"].rank()) ** 2) / (n * (n**2 - 1))
print(spirmam)
# ваш код: обчисліть обрану міру

['важкий' 'серед важ' 'вкрай важкий '] ['сердній' 'середній' 'критичний' nan 'важкий' 'легкий' 'серед ступ'
 'легка' 'вкрай важкий ' 'легкий ' 'вкрай важкий' 'важкий '
 'серед ступеня']
0.6985657564216838


In [13]:
# Тепер — контрприклад: застосуйте кореляцію Пірсона незалежно від шкал
# і поясніть, чому отримане число не можна використовувати.
pdiff = dataSpirman["Перебіг"] - dataSpirman["Перебіг"].mean()
sdiff = dataSpirman["ступінь КТ"] - dataSpirman["ступінь КТ"].mean()
# print(sum(pdiff), (sdiff))
pirson = np.sum(pdiff * sdiff) / np.sqrt(np.sum(pdiff**2) * np.sum(sdiff**2))
print(pirson)

0.6281471223681993


резульатт звісно вийшов схожим але пірсон працює з значенями вважаючи їх однієї шкали, в нас же «відстань» між I та II це не 1 це хто зна скільки на відміну від відстанню між 0.432 та 0.67 у Прокальцитоніну

---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

номер і/х	ідентифікатор	Номер історії хвороби лише позначає пацієнта. також є ду...
1	Діагноз	номінальна	Текстова назва діагнозу; категорії можна лише порівнюват...
2	ЛН	порядкова	Ступінь легеневої недостатності І < ІІ має природний пор...
3	Дата розтину	інтервальна	Дата — точка на шкалі часу з нулем у початку епохи, різн...
4	КТ	відношень	Компютерна томографія легень??? ураження? має нуль, відн...
5	Прокальцитонін	відношень	має фізичний нуль — відсутність речовини, відношення зна...
6	БРзаг2	відношень	Загальний білірубін (мкмоль/л)и тн57го — концентрація з ...
7	МНВ1	

In [14]:
ПІДГОТОВКА = {
    # 'стовпець': 'спосіб і коротке обґрунтування',
    "номер і/х": "не брати, позначає пацієнта і не несе для моделі інформації",
    "Діагноз": "звести різні написання до 2–4 канонічних, потім one-hot",
    "ЛН": "впорядкована категорія: коди 1-6",
    "Дата розтину": "перехід до тривалостей: кількість днів від поступлення. сама дата як точка нічого не дає моделі, а тривалість уже шкала відношень",
    "КТ": "привести змішаний запис до єдиної частки ураження, потім стандартизація",
    "Прокальцитонін": "логарифмування, потім стандартизація. розподіл сильно скошений вправо",
    "БРзаг2": "логарифмування, потім стандартизація. розподіл скошений вправо, треба щоб ознаки мали співмірний масштаб",
    "МНВ1": "стандартизація, щоб масштаб не впливав на вагу ознаки в моделі",
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, "")

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ], columns=["Стовпець", "Шкала", "Як готуємо"])

,Стовпець,Шкала,Як готуємо
0,номер і/х,ідентифікатор,"не брати, позначає пацієнта і не несе для моделі інформації"
1,Діагноз,номінальна,"звести різні написання до 2–4 канонічних, потім one-hot"
2,ЛН,порядкова,впорядкована категорія: коди 1-6
3,Дата розтину,інтервальна,перехід до тривалостей: кількість днів від поступлення. ...
4,КТ,відношень,"привести змішаний запис до єдиної частки ураження, потім..."
5,Прокальцитонін,відношень,"логарифмування, потім стандартизація. розподіл сильно ск..."
6,БРзаг2,відношень,"логарифмування, потім стандартизація. розподіл скошений ..."
7,МНВ1,відношень,"стандартизація, щоб масштаб не впливав на вагу ознаки в ..."


In [15]:
# ваш код: реалізуйте підготовку для трьох стовпців різних шкал
prepared = pd.DataFrame(index=df.index)


def canonical_diagnosis(text):
    t = str(text).lower().replace(" ", "")
    group = "3кл" if "3кл" in t else "4кл" if "4кл" in t else "інше"
    side = "двобічна" if "дв" in t else "бік не вказано"
    return f"полісегм. пневмонія {group}, {side}"


diagnosis = df["Діагноз"].map(canonical_diagnosis)
print("Діагноз: ", diagnosis.nunique(), "категорії")
print(diagnosis.value_counts())
prepared = prepared.join(pd.get_dummies(diagnosis, prefix="Діагноз", dtype=int))

days = (df["Дата розтину"] - pd.to_datetime(df["Поступлення"])).dt.days
invalid = days < 0
print("Дата розтину раніше за поступлення:")
print(df.loc[invalid, ["номер і/х", "Поступлення", "Дата розтину"]])
prepared["днів_до_розтину"] = days.mask(invalid)

pct = df["Прокальцитонін"]
print(f"Прокальцитонін: скос {pct.skew()}")
log_pct = np.log(pct)
prepared["Прокальцитонін_log_z"] = (log_pct - log_pct.mean()) / log_pct.std()

prepared.head(5)

Діагноз:  4 категорії
Діагноз
полісегм. пневмонія 4кл, двобічна          86
полісегм. пневмонія 3кл, двобічна          26
полісегм. пневмонія 4кл, бік не вказано     5
полісегм. пневмонія 3кл, бік не вказано     4
Name: count, dtype: int64
Дата розтину раніше за поступлення:
   номер і/х Поступлення Дата розтину
81      2551  2022-03-26   2022-02-03
90      3232  2021-12-15   2021-01-06
Прокальцитонін: скос 5.202167922066669


,"Діагноз_полісегм. пневмонія 3кл, бік не вказано","Діагноз_полісегм. пневмонія 3кл, двобічна","Діагноз_полісегм. пневмонія 4кл, бік не вказано","Діагноз_полісегм. пневмонія 4кл, двобічна",днів_до_розтину,Прокальцитонін_log_z
0,0,0,0,1,15.0,-0.125267
1,0,0,1,0,NaN,NaN
2,0,0,1,0,NaN,NaN
3,0,0,0,1,NaN,NaN
4,0,0,0,1,NaN,NaN


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [16]:
for c in МОЇ:
    nna = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(nna) and nna.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(nna) and nna.max() == nna.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [17]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [18]:
ВИПРАВЛЕННЯ = [
    # (розділ, 'що не так', 'чому число неправильне', 'як правильно'),
    (
        1,
        "середнє від номінальної змінної помер/жив (коди 1 і 2)",
        "1 і 2 - це просто мітки, «посередині між смертю і життям» не буває; 1.50 нічого не означає",
        "рахувати частки: померло 50.4%, вижило 49.6%",
    ),
    (
        2,
        "не об'єднані різні написання однієї категорії (ж/жін, ч/чол)",
        "nunique() рахує написання. 4 статі це через брудність даних",
        "взяти першу літеру після strip(): 2 статі, жінок 75, чоловіків 46",
    ),
    (
        3,
        "середнє від порядкової шкали + сортування за частотою замість тяжкості + дублі написань",
        "коди присвоєні за частотою, тому середнє 8.29 -> «легкий» - це позиція в рейтингу частот, а не ступінь ураження",
        "звести написання, задати порядок легкий<середній<важкий<критичний<вкрай важкий, рахувати медіану/моду - «середній»",
    ),
    (
        4,
        'to_numeric(errors="coerce") мовчки викидає текстові значення КТ',
        '16 з 43 значень записані текстом ("25-30%", "до 17%") і стали NaN',
        "привести все до %. по 43 пацієнтах середнє ≈ 42.5%, медіана 35%",
    ),
    (
        5,
        "одне значення сатурації записане часткою (0.9) замість відсотків",
        "0.9 пораховано як 0.9%, тому мінімум 0.9 безглуздий, а середнє 82.05 занижене",
        "привести до відсотків (0.9 -> 90): середнє 82.79, мінімум 50",
    ),
    (
        6,
        "коефіцієнт варіації пораховано для дат",
        "CV ділить на середнє від умовного нуля епохи. насправді госпіталізації дуже нерівномірні від 1 до 30 на місяць",
        "описувати розкид у днях і кількістю госпіталізацій за місяць - пік у квітні 2022",
    ),
    (
        7,
        "Пірсон між номінальними змінними",
        "знак r залежить від довільного кодування 0/1, а 0.177 - не нуль: летальність жінок 57% проти 39% у чоловіків",
        "Х² і V Крамера ≈ 0.18 - слабкий, але наявний зв'язок",
    ),
    (
        8,
        "нормалізовано всі числові стовпці підряд, разом із ціллю і майже порожніми",
        "помер/жив - номінальна ціль, 1/2 це мітки; у ФЕРИТ2 одне значення",
        "не чіпати ціль, прибрати стовпці з багатьма пропусками, нормалізувати решту",
    ),
    (
        9,
        "dropna() по всіх 77 стовпцях",
        "в кожному рядку є хоч один пропуск, тому лишається 0 рядків",
        "спершу викинути стовпці з >30% пропусків, потім рядки: 58 стовпців і 52 пацієнти",
    ),
    (
        10,
        "дата розтину як предиктор летальності",
        "розтин є тільки в померлих - це витік цілі; ІЛ6-2 з r = 0.62 лише на 6 пацієнтах",
        "прибрати дату розтину і ціль, брати ознаки, відомі хоча б для 30 пацієнтів: топ - Лім2 0.645, Л2 0.48, Вік 0.45",
    ),
]

assert len(ВИПРАВЛЕННЯ) == 10, f"Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10"
for p in ВИПРАВЛЕННЯ:
    assert len(p) == 4 and all(str(x).strip() for x in p), "Заповніть усі чотири поля"
    assert len(str(p[2])) >= 30 and len(str(p[3])) >= 25, "Пояснення закороткі"

pd.DataFrame(ВИПРАВЛЕННЯ, columns=["Розділ", "Що не так", "Чому число неправильне", "Як правильно"])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,середнє від номінальної змінної помер/жив (коди 1 і 2),"1 і 2 - це просто мітки, «посередині між смертю і життям...","рахувати частки: померло 50.4%, вижило 49.6%"
1,2,"не об'єднані різні написання однієї категорії (ж/жін, ч/...",nunique() рахує написання. 4 статі це через брудність даних,"взяти першу літеру після strip(): 2 статі, жінок 75, чол..."
2,3,середнє від порядкової шкали + сортування за частотою за...,"коди присвоєні за частотою, тому середнє 8.29 -> «легкий...","звести написання, задати порядок легкий<середній<важкий<..."
3,4,"to_numeric(errors=""coerce"") мовчки викидає текстові знач...","16 з 43 значень записані текстом (""25-30%"", ""до 17%"") і ...","привести все до %. по 43 пацієнтах середнє ≈ 42.5%, меді..."
4,5,одне значення сатурації записане часткою (0.9) замість в...,"0.9 пораховано як 0.9%, тому мінімум 0.9 безглуздий, а с...","привести до відсотків (0.9 -> 90): середнє 82.79, мініму..."
5,6,коефіцієнт варіації пораховано для дат,CV ділить на середнє від умовного нуля епохи. насправді ...,описувати розкид у днях і кількістю госпіталізацій за мі...
6,7,Пірсон між номінальними змінними,"знак r залежить від довільного кодування 0/1, а 0.177 - ...","Х² і V Крамера ≈ 0.18 - слабкий, але наявний зв'язок"
7,8,"нормалізовано всі числові стовпці підряд, разом із ціллю...","помер/жив - номінальна ціль, 1/2 це мітки; у ФЕРИТ2 одне...","не чіпати ціль, прибрати стовпці з багатьма пропусками, ..."
8,9,dropna() по всіх 77 стовпцях,"в кожному рядку є хоч один пропуск, тому лишається 0 рядків","спершу викинути стовпці з >30% пропусків, потім рядки: 5..."
9,10,дата розтину як предиктор летальності,розтин є тільки в померлих - це витік цілі; ІЛ6-2 з r = ...,"прибрати дату розтину і ціль, брати ознаки, відомі хоча ..."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [19]:
# --- 1. результат лікування ---
print('1. НЕПРАВИЛЬНО: середнє =', round(df['помер/жив'].mean(), 3))
print('1. ПРАВИЛЬНО  :')
print(df['помер/жив'].map({1: 'помер', 2: 'вижив'}).value_counts(normalize=True).round(3).to_string())
print()

# --- 2. стать ---
print('2. НЕПРАВИЛЬНО: статей =', df['Стать'].nunique())
стать = df['Стать'].str.strip().str[0]          # 'жін' -> 'ж', 'чол' -> 'ч'
print('2. ПРАВИЛЬНО  : статей =', стать.nunique())
print(стать.value_counts().to_string())
print()

# --- 9. пропуски ---
print('9. НЕПРАВИЛЬНО: після dropna() лишилось рядків =', len(df.dropna()))
чистий = df.loc[:, df.isna().mean() <= 0.3].dropna()   # спершу стовпці з >30% пропусків, потім рядки
print('9. ПРАВИЛЬНО  : лишилось', len(чистий), 'рядків і', чистий.shape[1], 'стовпців')

1. НЕПРАВИЛЬНО: середнє = 1.496
1. ПРАВИЛЬНО  :
помер/жив
помер    0.504
вижив    0.496

2. НЕПРАВИЛЬНО: статей = 4
2. ПРАВИЛЬНО  : статей = 2
Стать
ж    75
ч    46

9. НЕПРАВИЛЬНО: після dropna() лишилось рядків = 0
9. ПРАВИЛЬНО  : лишилось 52 рядків і 58 стовпців


---
# Етап 7. Висновки та індивідуальне питання

In [20]:
ВИСНОВКИ = [
    # три висновки, кожен — повне речення про ваші дані, а не загальна фраза
    'За хибних припущень або неправильних статистик як це було у неправильному звіті із наприклад живий/мертвий можна отримати нісенітниці по типу пацієнт в середньому напів живий',
    'Дані брудні: Діагноз має 15 написань лише для 4 реальних діагнозів, КТ записано то часткою, то текстом («25-30%», «до 17%»), а у двох пацієнтів дата розтину раніша за дату поступлення.',
    'Перебіг і ступінь КТ (обидві порядкові) помітно пов’язані: Спірмен 0.70 на 42 пацієнтах; Пірсон дає 0.63, але він вважає відстані між рівнями однаковими, тому його число інтерпретувати не можна.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. За хибних припущень або неправильних статистик як це було у неправильному звіті із наприклад живий/мертвий можна отримати нісенітниці по типу пацієнт в середньому напів живий
2. Дані брудні: Діагноз має 15 написань лише для 4 реальних діагнозів, КТ записано то часткою, то текстом («25-30%», «до 17%»), а у двох пацієнтів дата розтину раніша за дату поступлення.
3. Перебіг і ступінь КТ (обидві порядкові) помітно пов’язані: Спірмен 0.70 на 42 пацієнтах; Пірсон дає 0.63, але він вважає відстані між рівнями однаковими, тому його число інтерпретувати не можна.


In [21]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = (
    'pandas спокійно порахує для помер/живий середнє 1.50, стандартне відхилення 0.50 і медіану 1.0, бо бачить лише числа. Але ці числа є просто мітками, бо «помер» і «вижив» — номінальна шкала. Середнє 1.50 означало б пацієнта посередині між смертю і життям, нещасний кіт шркденгера не пацієнт, такого ящко ми досліджуємо нормальні обставини, а вони такими є, не буває. Стандартне відхилення 0.50 — це «відстань» між мітками, можна спокійно і валідно змінити кодування на нехай -12 і 12 і це буде ок тільки не для відхилення, хоча пацієнти ті самі. Медіана 1.0 вимагає порядку, а «помер» не менше і не більше за «вижив»; вона лише випадково збіглася з модою, бо померлих на одного більше.'
)
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Стовпець «помер/жив» містить числа 1 і 2. Які три різні статистики pandas порахує для нього без жодної скарги — і чому кожна з них безглузда?

pandas спокійно порахує для помер/живий середнє 1.50, стандартне відхилення 0.50 і медіану 1.0, бо бачить лише числа. Але ці числа є просто мітками, бо «помер» і «вижив» — номінальна шкала. Середнє 1.50 означало б пацієнта посередині між смертю і життям, нещасний кіт шркденгера не пацієнт, такого ящко ми досліджуємо нормальні обставини, а вони такими є, не буває. Стандартне відхилення 0.50 — це «відстань» між мітками, можна спокійно і валідно змінити кодування на нехай -12 і 12 і це буде ок тільки не для відхилення, хоча пацієнти ті самі. Медіана 1.0 вимагає порядку, а «помер» не менше і не більше за «вижив»; вона лише випадково збіглася з модою, бо померлих на одного більше.


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [22]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'ні',
    'етап 2 — допустимі статистики':   'ні',
    'етап 3 — безглузді числа':        'ні',
    'етап 4 — міра зв’язку':           'синтаксис',
    'етап 5 — підготовка до моделі':   'синтаксис',
    'етап 6 — виправлення аналізу':    'текст, перероблено',
    'етап 7 — висновки й відповідь':   'текст, перероблено',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'я самостійно дойшов до 6го блоку неправильного звіту, далі я передивився виправлення і поміняв трохи очищення набору від пропусків та зв\'зку статі із результатом лікування, де мені не сподобалися рішення аі, помилки звіту виправляв із запитуванням синтаксису'
ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [23]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 13 готова до захисту.
Fediv Andrii, ШІ-34


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.